# Hackathon 9: Transfer Learning y stacking en Fast-Food-101

Se evalúan ResNet50, DenseNet121 y MobileNetV2 con cinco folds estratificados. En cada fold exterior se reserva una parte estratificada del entrenamiento para ajustar la regresión logística; el fold exterior no participa en el entrenamiento de los modelos ni del meta-modelo. Las métricas son macro (cada clase pesa igual). El experimento usa los 6.000 archivos del ZIP suministrado.

### Configuración e imports

In [ ]:
import os, json, random, zipfile, traceback, time, copy
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from IPython.display import display
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark=True
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT=Path('/kaggle/working'); OUT.mkdir(exist_ok=True)
progress={'status':'running','stage':'setup','device':str(DEVICE),'artifacts':[]}
def save(): (OUT/'run_summary.json').write_text(json.dumps(progress,indent=2,ensure_ascii=False),encoding='utf-8')
def stage(name, fn):
    if progress['status']=='failed': return
    progress['stage']=name; save()
    try: fn()
    except Exception as e:
        progress.update(status='failed',error=repr(e),traceback=traceback.format_exc()); save(); print(progress['traceback'])
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
save()

### Inspección de imágenes y balance
Las imágenes provienen del archivo proporcionado; la estratificación conserva la proporción por clase.

In [ ]:
CLASSES=['chicken_wings','hamburger','hot_dog','pizza','sushi','tacos']
def inspect_data():
    global frame
    assert DEVICE.type=='cuda', 'Se requiere GPU T4'
    assert 'T4' in torch.cuda.get_device_name(0), 'La GPU asignada no es T4'
    root=Path('/kaggle/input'); archives=list(root.rglob('fast_food.zip'))
    dest=Path('/tmp/hack9_images')
    if archives:
        with zipfile.ZipFile(archives[0]) as z: z.extractall(dest)
    else: dest=root
    paths=[]
    for label in CLASSES:
        dirs=[x for x in dest.rglob(label) if x.is_dir()]
        assert len(dirs)==1, (label,dirs)
        paths.extend((str(f),label) for f in sorted(dirs[0].glob('*')) if f.suffix.lower() in ('.jpg','.jpeg','.png'))
    frame=pd.DataFrame(paths,columns=['path','label']); frame['target']=frame.label.map({x:i for i,x in enumerate(CLASSES)})
    assert frame.groupby('label').size().min()>=5
    print('Imágenes:',len(frame)); print(frame.groupby('label').size().to_string())
    fig,axes=plt.subplots(1,6,figsize=(16,3))
    for ax,label in zip(axes,CLASSES):
        ax.imshow(Image.open(frame[frame.label==label].path.iloc[0]).convert('RGB')); ax.set_title(label); ax.axis('off')
    fig.tight_layout(); fig.savefig(OUT/'ejemplos.png',dpi=120); plt.show()
    frame.groupby('label').size().to_csv(OUT/'balance.csv',header=['imagenes'])
stage('datos',inspect_data)

### Preprocesamiento y aumento
Se normaliza con las estadísticas de ImageNet. En entrenamiento, recorte aleatorio, giro horizontal y cambios leves de color ayudan a tolerar escala, orientación e iluminación. Validación usa un recorte central determinista y nunca recibe aumentos.

In [ ]:
train_tf=transforms.Compose([transforms.RandomResizedCrop(224,scale=(0.75,1.0)),transforms.RandomHorizontalFlip(),transforms.ColorJitter(0.15,0.15,0.15,0.05),transforms.ToTensor(),transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
valid_tf=transforms.Compose([transforms.Resize(256),transforms.CenterCrop(224),transforms.ToTensor(),transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
class Food(Dataset):
    def __init__(self,rows,tf): self.rows=rows.reset_index(drop=True); self.tf=tf
    def __len__(self): return len(self.rows)
    def __getitem__(self,i):
        row=self.rows.iloc[i]
        with Image.open(row.path) as img: x=self.tf(img.convert('RGB'))
        return x,int(row.target)
def loader(indices,training=False):
    return DataLoader(Food(frame.iloc[indices],train_tf if training else valid_tf),batch_size=32,shuffle=training,num_workers=2,pin_memory=True)
def create_model(name):
    if name=='ResNet50':
        m=models.resnet50(weights=models.ResNet50_Weights.DEFAULT); m.fc=nn.Linear(m.fc.in_features,6); head=m.fc; tail=m.layer4
    elif name=='DenseNet121':
        m=models.densenet121(weights=models.DenseNet121_Weights.DEFAULT); m.classifier=nn.Linear(m.classifier.in_features,6); head=m.classifier; tail=m.features.denseblock4
    else:
        m=models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT); m.classifier[1]=nn.Linear(m.classifier[1].in_features,6); head=m.classifier; tail=m.features[-3:]
    for p in m.parameters(): p.requires_grad=False
    for p in list(head.parameters())+list(tail.parameters()): p.requires_grad=True
    return m.to(DEVICE)

### Entrenamiento y predicciones
Se descongela la cabeza y el último bloque de cada red para realizar fine-tuning; se usa AdamW, tasa 0,0001, cuatro épocas y entropía cruzada. El meta-conjunto es el 20 % del entrenamiento externo: sus imágenes no entrenan las redes base. Esto evita usar predicciones dentro de muestra para entrenar el meta-modelo. El costo es entrenar cada red con el 64 % del conjunto completo.

In [ ]:
NAMES=['ResNet50','DenseNet121','MobileNetV2']; EPOCHS=4
history=[]; metrics=[]; all_predictions=[]; meta_rows=[]
def train_model(name,fold,train_idx,meta_idx,test_idx):
    model=create_model(name); optimizer=torch.optim.AdamW(filter(lambda p:p.requires_grad,model.parameters()),lr=1e-4,weight_decay=1e-4)
    scaler=torch.amp.GradScaler('cuda'); loss_fn=nn.CrossEntropyLoss()
    for epoch in range(EPOCHS):
        model.train(); total=correct=loss_sum=0
        for x,y in loader(train_idx,True):
            x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True); optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda',dtype=torch.float16): logits=model(x); loss=loss_fn(logits,y)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            total+=len(y); loss_sum+=loss.item()*len(y); correct+=(logits.argmax(1)==y).sum().item()
        model.eval(); val_loss=val_correct=val_total=0
        with torch.no_grad():
            for x,y in loader(meta_idx):
                x=x.to(DEVICE); y=y.to(DEVICE)
                with torch.autocast(device_type='cuda',dtype=torch.float16): logits=model(x); loss=loss_fn(logits,y)
                val_total+=len(y); val_loss+=loss.item()*len(y); val_correct+=(logits.argmax(1)==y).sum().item()
        history.append(dict(fold=fold,model=name,epoch=epoch+1,train_loss=loss_sum/total,train_accuracy=correct/total,meta_loss=val_loss/val_total,meta_accuracy=val_correct/val_total))
        pd.DataFrame(history).to_csv(OUT/'historial.csv',index=False)
        print(f'{name} fold {fold} epoch {epoch+1}: train loss {loss_sum/total:.3f}, meta loss {val_loss/val_total:.3f}')
    def predict(indices):
        outputs=[]; model.eval()
        with torch.no_grad():
            for x,_ in loader(indices):
                with torch.autocast(device_type='cuda',dtype=torch.float16): outputs.append(model(x.to(DEVICE)).float().softmax(1).cpu().numpy())
        return np.concatenate(outputs)
    a,b=predict(meta_idx),predict(test_idx)
    del model,optimizer; torch.cuda.empty_cache()
    return a,b
def score(y,p,fold,name):
    metrics.append(dict(fold=fold,model=name,precision_macro=precision_score(y,p,average='macro',zero_division=0),f1_macro=f1_score(y,p,average='macro',zero_division=0)))

### Validación cruzada y stacking
Para cada fold externo se ajusta una regresión logística sobre las probabilidades de las tres redes en el conjunto interno de meta-entrenamiento. Se compara con los modelos individuales y con el promedio simple de probabilidades sobre exactamente las mismas imágenes externas. No se supone que stacking siempre mejore.

In [ ]:
def experiment():
    outer=StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED); y=frame.target.to_numpy()
    for fold,(outer_train,test_idx) in enumerate(outer.split(np.zeros(len(y)),y),1):
        train_idx,meta_idx=train_test_split(outer_train,test_size=0.2,stratify=y[outer_train],random_state=SEED+fold)
        assert not (set(train_idx)&set(meta_idx) or set(train_idx)&set(test_idx) or set(meta_idx)&set(test_idx))
        meta_probs=[]; test_probs=[]
        for name in NAMES:
            progress.update(stage='entrenamiento',fold=fold,model=name); save()
            a,b=train_model(name,fold,train_idx,meta_idx,test_idx); meta_probs.append(a); test_probs.append(b)
            score(y[test_idx],b.argmax(1),fold,name)
        meta_x=np.concatenate(meta_probs,axis=1); test_x=np.concatenate(test_probs,axis=1)
        meta=LogisticRegression(max_iter=2000,random_state=SEED).fit(meta_x,y[meta_idx]); pred=meta.predict(test_x)
        score(y[test_idx],pred,fold,'Stacking'); score(y[test_idx],np.mean(test_probs,axis=0).argmax(1),fold,'Promedio')
        for i,idx in enumerate(test_idx):
            row={'fold':fold,'index':int(idx),'path':frame.path.iloc[idx],'truth':int(y[idx]),'Stacking':int(pred[i]),'Promedio':int(np.mean(test_probs,axis=0)[i].argmax())}
            row.update({name:int(probs[i].argmax()) for name,probs in zip(NAMES,test_probs)}); all_predictions.append(row)
        columns=[f'{name}_{label}' for name in NAMES for label in CLASSES]
        mf=pd.DataFrame(meta_x,columns=columns); mf.insert(0,'target',y[meta_idx]); mf.insert(0,'fold',fold); meta_rows.append(mf)
        pd.DataFrame(metrics).to_csv(OUT/'metricas_por_fold.csv',index=False)
        pd.DataFrame(all_predictions).to_csv(OUT/'predicciones.csv',index=False)
        pd.concat(meta_rows,ignore_index=True).to_csv(OUT/'dataset_meta.csv',index=False)
        print('Fold terminado:',fold)
stage('experimento',experiment)

### Comparación, matrices y curvas
La desviación estándar entre folds describe variación de esta partición, no un intervalo de confianza. Las curvas de entrenamiento frente a meta-validación ayudan a detectar sobreajuste; la diferencia entre stacking y el mejor modelo es descriptiva, no una prueba de significancia.

In [ ]:
def report():
    m=pd.DataFrame(metrics); summary=m.groupby('model')[['precision_macro','f1_macro']].agg(['mean','std']).sort_values(('f1_macro','mean'),ascending=False)
    summary.to_csv(OUT/'resumen_metricas.csv'); display(summary)
    pred=pd.DataFrame(all_predictions); fig,axes=plt.subplots(1,5,figsize=(24,4))
    for ax,name in zip(axes,NAMES+['Promedio','Stacking']):
        ConfusionMatrixDisplay(confusion_matrix(pred.truth,pred[name],labels=range(6)),display_labels=CLASSES).plot(ax=ax,colorbar=False,xticks_rotation=90); ax.set_title(name)
    fig.tight_layout(); fig.savefig(OUT/'matrices_confusion.png',dpi=130); plt.show()
    h=pd.DataFrame(history); fig,axes=plt.subplots(1,3,figsize=(15,4))
    for ax,name in zip(axes,NAMES):
        g=h[h.model==name].groupby('epoch')[['train_loss','meta_loss']].mean(); g.plot(ax=ax); ax.set_title(name); ax.set_ylabel('Pérdida media'); ax.set_xlabel('Época')
    fig.tight_layout(); fig.savefig(OUT/'curvas_entrenamiento.png',dpi=130); plt.show()
    base=summary.loc[NAMES].sort_values(('f1_macro','mean'),ascending=False); best=base.index[0]
    delta=float(summary.loc['Stacking',('f1_macro','mean')]-summary.loc[best,('f1_macro','mean')])
    cm=confusion_matrix(pred.truth,pred.Stacking,labels=range(6)); errors=cm.sum(axis=1)-np.diag(cm)
    complement={f'{a} + {b}':float(((pred[a]==pred.truth)|(pred[b]==pred.truth)).mean()) for i,a in enumerate(NAMES) for b in NAMES[i+1:]}
    analysis={'mejor_individual':best,'diferencia_f1_stacking_menos_mejor':delta,'errores_stacking_por_clase':dict(zip(CLASSES,errors.tolist())),'acierto_oraculo_por_pareja':complement,'nota':'El oráculo es un límite descriptivo: conoce si cada modelo acierta. No es un ensemble implementable ni evidencia causal.'}
    (OUT/'analisis.json').write_text(json.dumps(analysis,indent=2,ensure_ascii=False),encoding='utf-8'); print(json.dumps(analysis,indent=2,ensure_ascii=False))
    progress.update(status='complete',stage='done',artifacts=[x.name for x in OUT.iterdir() if x.is_file()]); save()
stage('informe',report)
print('Estado:',progress['status'], 'etapa:',progress['stage'])